# Thêm Thư Viện

In [1]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [2]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)
conn_dwh_lib = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=dwh_lib;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)

# Đọc data

In [3]:
# Hàm đọc dữ liệu từng phần và xử lý lỗi
def fetch_data_in_batches(query_base, connection, batch_size=100):
    offset = 0
    all_data = []  # Lưu tất cả các hàng hợp lệ
    while True:
        query = f"""
        {query_base}
        ORDER BY ID
        OFFSET {offset} ROWS FETCH NEXT {batch_size} ROWS ONLY
        """
        try:
            # Đọc dữ liệu batch hiện tại
            df_batch = pd.read_sql(query, connection)
            if df_batch.empty:  # Nếu không còn dữ liệu, dừng vòng lặp
                break
            all_data.append(df_batch)  # Lưu batch hợp lệ
            offset += batch_size  # Tăng offset để đọc batch tiếp theo
        except Exception as e:
            print(f"Lỗi xảy ra khi xử lý batch từ {offset}: {e}")
            offset += batch_size  # Bỏ qua batch bị lỗi và tiếp tục
    # Gộp tất cả các batch thành DataFrame duy nhất
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

## Đọc bảng An_pham_cho_muon

In [4]:
query_Anphamchomuon = """
SELECT Tai_lieu_ID,
       Ma_xep_gia, 
       So_the_ID,
       Ngay_muon,
       Ngay_tra,
       So_luot_gia_han,
       Note
  FROM An_pham_cho_muon
  WHERE YEAR(Ngay_muon) = 2002
"""
df_apcm = fetch_data_in_batches(query_Anphamchomuon, conn_libol, batch_size=100) # Gọi hàm để lấy dữ liệu
print(df_apcm)

   Tai_lieu_ID Ma_xep_gia  So_the_ID           Ngay_muon            Ngay_tra  \
0          665  skv003743       3060 2002-10-22 23:31:00 2003-06-06 11:20:11   
1          701  skv003812       3060 2002-10-22 23:32:00 2003-06-06 11:20:44   
2          820  skv003913       3060 2002-10-22 23:37:00 2003-06-06 11:25:40   
3          876  SKV005496       3364 2002-11-10 20:22:00 2003-06-25 09:09:50   
4          892  SKV006782       2825 2002-11-11 13:47:00 2003-04-01 13:41:54   
5         1724  SKV011070       5208 2002-11-27 21:24:00 2003-04-18 10:10:47   
6         1886  SKV010191       3856 2002-12-06 07:23:00 2003-04-26 07:16:54   
7          519  SKV001889       4291 2002-12-06 14:46:00 2003-05-03 00:00:00   
8         3307  SKV011983       1712 2002-12-19 18:11:00 2003-05-10 09:04:43   
9          734  SKV005519       3060 2002-12-25 01:01:00 2003-08-06 15:54:40   

   So_luot_gia_han  Note  
0                0  None  
1                0  None  
2                0  None  
3          

C:\Users\admin\AppData\Local\Temp\ipykernel_6480\3076783788.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_batch = pd.read_sql(query, connection)


## Đọc bảng Lich_su_muon_sach

In [5]:
query_Lichsumuonsach = """
SELECT Tai_lieu_ID,
       Ma_xep_gia, 
       So_the_ID,
       Ngay_muon,
       Ngay_tra,
       So_ngay_qua_han,
       Tien_phat
  FROM Lich_su_muon_sach
  WHERE YEAR(Ngay_muon) = 2002
"""
df_lscm = fetch_data_in_batches(query_Lichsumuonsach, conn_libol, batch_size=100) # Gọi hàm để lấy dữ liệu
print(df_lscm)

C:\Users\admin\AppData\Local\Temp\ipykernel_6480\3076783788.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_batch = pd.read_sql(query, connection)


       Tai_lieu_ID Ma_xep_gia  So_the_ID           Ngay_muon  \
0              356  SKV001684        9.0 2002-09-09 15:03:00   
1              356  SKV001684        9.0 2002-09-09 15:39:00   
2              295  SKV000804        9.0 2002-09-09 15:05:00   
3              356  SKV001684        9.0 2002-09-09 15:44:00   
4              356  SKV001684        9.0 2002-09-09 15:48:00   
...            ...        ...        ...                 ...   
23962        36917  GT0328618        NaN 2002-01-01 00:00:00   
23963        36914  GT0284977        NaN 2002-01-01 00:00:00   
23964        20277  GT0139710        NaN 2002-01-01 00:00:00   
23965        15476  GT0083078    77634.0 2002-01-01 00:00:00   
23966        38734  GT0316259        NaN 2002-01-01 00:00:00   

                 Ngay_tra  So_ngay_qua_han  Tien_phat  
0     2002-09-09 15:04:00                0        0.0  
1     2002-09-09 15:43:00                0        0.0  
2     2002-09-09 15:44:00                0        0.0  
3     2

# Xử lý data

## Xử lý cột còn thiếu cho 2 bảng

In [6]:
# thêm 2 cột còn thiếu vào 
df_apcm['So_ngay_qua_han'] = None
df_apcm['Tien_phat'] = None
# chỉnh sửa cho ngày trả là None hết vì chưa trả sách
df_apcm['Ngay_tra'] = None

df_lscm['So_luot_gia_han'] = None
df_lscm['Note'] = None

print(df_apcm)
print(df_lscm)

   Tai_lieu_ID Ma_xep_gia  So_the_ID           Ngay_muon Ngay_tra  \
0          665  skv003743       3060 2002-10-22 23:31:00     None   
1          701  skv003812       3060 2002-10-22 23:32:00     None   
2          820  skv003913       3060 2002-10-22 23:37:00     None   
3          876  SKV005496       3364 2002-11-10 20:22:00     None   
4          892  SKV006782       2825 2002-11-11 13:47:00     None   
5         1724  SKV011070       5208 2002-11-27 21:24:00     None   
6         1886  SKV010191       3856 2002-12-06 07:23:00     None   
7          519  SKV001889       4291 2002-12-06 14:46:00     None   
8         3307  SKV011983       1712 2002-12-19 18:11:00     None   
9          734  SKV005519       3060 2002-12-25 01:01:00     None   

   So_luot_gia_han  Note So_ngay_qua_han Tien_phat  
0                0  None            None      None  
1                0  None            None      None  
2                0  None            None      None  
3                0  None    

## Gộp 2 bảng lại

In [ ]:
df_phieumuon = pd.concat([df_lscm, df_apcm], ignore_index=True)
df_phieumuon = df_phieumuon.sort_values(by='Ngay_muon', ascending=True).reset_index(drop=True) # sắp xếp lại cho dễ nhìn

query_MaxID = "SELECT MAX(ID_phieu_muon) AS MaxID FROM FACT_Phieu_muon_sach" # Lấy giá trị MaxID từ bảng FACT_Phieu_muon_sach
df_MaxID = pd.read_sql(query_MaxID, conn_dwh_lib)
max_id = int(df_MaxID['MaxID'].iloc[0]) if not df_MaxID.empty else 0 # Giá trị khởi tạo ID mới, bắt đầu từ MaxID + 1
start_id = max_id + 1
df_phieumuon.insert(0, 'ID', range(start_id, start_id + len(df_phieumuon))) # Thêm cột ID mới đếm từ MaxID + 1


df_phieumuon.rename(columns={'Tai_lieu_ID': 'ID_tai_lieu'}, inplace=True)
print(df_phieumuon)

          ID  ID_tai_lieu Ma_xep_gia  So_the_ID  Ngay_muon  \
0         11        17740  GT0096888        NaN 2002-01-01   
1         12        36792  GT0329693        NaN 2002-01-01   
2         13        37389  GT0304620    72024.0 2002-01-01   
3         14        28177  GTD013720    72024.0 2002-01-01   
4         15        36920  GT0286764    72024.0 2002-01-01   
...      ...          ...        ...        ...        ...   
23972  23983          285  SKV000569     1323.0 2002-12-31   
23973  23984          571  SKV002765        NaN 2002-12-31   
23974  23985          405  SKV002298        NaN 2002-12-31   
23975  23986         2179  skv013003        NaN 2002-12-31   
23976  23987          892  SKV006755        NaN 2002-12-31   

                 Ngay_tra So_ngay_qua_han  Tien_phat So_luot_gia_han  Note  
0     2016-06-17 14:18:00            4981  2490500.0            None  None  
1     2016-04-27 09:10:00            4930  2465000.0            None  None  
2     2016-03-08 09:38:0

C:\Users\admin\AppData\Local\Temp\ipykernel_6480\3588075772.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_phieumuon = pd.concat([df_lscm, df_apcm], ignore_index=True)
C:\Users\admin\AppData\Local\Temp\ipykernel_6480\3588075772.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_MaxID = pd.read_sql(query_MaxID, conn_dwh_lib)


## Xử lý NaN và ""

In [9]:
df_phieumuon = df_phieumuon.replace('', None)
df_phieumuon = df_phieumuon.replace(np.nan, None)
print(df_phieumuon)

          ID  ID_tai_lieu Ma_xep_gia So_the_ID  Ngay_muon            Ngay_tra  \
0         11        17740  GT0096888      None 2002-01-01 2016-06-17 14:18:00   
1         12        36792  GT0329693      None 2002-01-01 2016-04-27 09:10:00   
2         13        37389  GT0304620   72024.0 2002-01-01 2016-03-08 09:38:00   
3         14        28177  GTD013720   72024.0 2002-01-01 2016-03-08 09:36:00   
4         15        36920  GT0286764   72024.0 2002-01-01 2016-03-08 08:39:00   
...      ...          ...        ...       ...        ...                 ...   
23972  23983          285  SKV000569    1323.0 2002-12-31 2003-01-24 14:08:00   
23973  23984          571  SKV002765      None 2002-12-31 2003-01-03 15:08:00   
23974  23985          405  SKV002298      None 2002-12-31 2003-01-08 08:45:00   
23975  23986         2179  skv013003      None 2002-12-31 2003-01-07 15:00:00   
23976  23987          892  SKV006755      None 2002-12-31 2003-01-10 08:28:00   

      So_ngay_qua_han  Tien

## Xử lý kiểu Date

In [10]:
query_date = "SELECT Date_key FROM DIM_Date"
df_date = pd.read_sql(query_date, conn_dwh_lib)
date_ids = set(df_date['Date_key'])
# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong date_key của bảng DIM_date hay không ?
df_phieumuon['Ngay_muon'] = pd.to_datetime(df_phieumuon['Ngay_muon'], errors='coerce')
df_phieumuon['Ngay_tra'] = pd.to_datetime(df_phieumuon['Ngay_tra'], errors='coerce')
df_phieumuon['Ngay_muon'] = df_phieumuon['Ngay_muon'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)
df_phieumuon['Ngay_tra'] = df_phieumuon['Ngay_tra'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)

print(df_phieumuon[['Ngay_muon', 'Ngay_tra']])

C:\Users\admin\AppData\Local\Temp\ipykernel_6480\3472876002.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_date = pd.read_sql(query_date, conn_dwh_lib)


       Ngay_muon  Ngay_tra
0       20020101  20160617
1       20020101  20160427
2       20020101  20160308
3       20020101  20160308
4       20020101  20160308
...          ...       ...
23972   20021231  20030124
23973   20021231  20030103
23974   20021231  20030108
23975   20021231  20030107
23976   20021231  20030110

[23977 rows x 2 columns]


## Xử lý ID_tai_lieu

In [11]:
query_Tailieu = "SELECT ID_tai_lieu FROM DIM_Tai_lieu"
df_tailieu = pd.read_sql(query_Tailieu, conn_dwh_lib)
tailieu_ids = set(df_tailieu['ID_tai_lieu'])
# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong ID_tai_lieu của bảng DIM_Tai_lieu hay không ?
df_phieumuon['ID_tai_lieu'] = df_phieumuon['ID_tai_lieu'].apply(lambda x: x if pd.notna(x) and x in tailieu_ids else 0)
print(df_phieumuon[['ID_tai_lieu']])

       ID_tai_lieu
0            17740
1            36792
2            37389
3            28177
4            36920
...            ...
23972          285
23973          571
23974          405
23975         2179
23976          892

[23977 rows x 1 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_6480\1236753618.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tailieu = pd.read_sql(query_Tailieu, conn_dwh_lib)


## Xử lý ID_xep_gia

In [12]:
query_Xepgia = "SELECT ID_xep_gia, ID_tai_lieu, Ma_xep_gia FROM DIM_Xep_gia"
df_xepgia = pd.read_sql(query_Xepgia, conn_dwh_lib)
# Gán ID_xep_gia từ df_xep_gia vào df_phieu_muon_sach
df_phieumuon['ID_xep_gia'] = None
df_phieumuon['ID_xep_gia'] = df_phieumuon.apply(lambda row: df_xepgia.loc[
                                                (df_xepgia['ID_tai_lieu'] == row['ID_tai_lieu']) & 
                                                (df_xepgia['Ma_xep_gia'] == row['Ma_xep_gia']), 
                                                'ID_xep_gia'
                                                ].iloc[0] if not df_xepgia[
                                                    (df_xepgia['ID_tai_lieu'] == row['ID_tai_lieu']) & 
                                                    (df_xepgia['Ma_xep_gia'] == row['Ma_xep_gia'])
                                                    ].empty else 0,
                                                    axis=1)
print(df_phieumuon[['ID_tai_lieu', 'Ma_xep_gia', 'ID_xep_gia']])

C:\Users\admin\AppData\Local\Temp\ipykernel_6480\1872470284.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_xepgia = pd.read_sql(query_Xepgia, conn_dwh_lib)


       ID_tai_lieu Ma_xep_gia  ID_xep_gia
0            17740  GT0096888      362738
1            36792  GT0329693      793449
2            37389  GT0304620      754181
3            28177  GTD013720      655585
4            36920  GT0286764      726661
...            ...        ...         ...
23972          285  SKV000569      182735
23973          571  SKV002765      185494
23974          405  SKV002298      184275
23975         2179  skv013003           0
23976          892  SKV006755      198631

[23977 rows x 3 columns]


## Xử lý ID_ban_doc

### Đọc từ libol để đổi ID sang So_the

In [171]:
query_Ban_doc = "SELECT ID, dbo.DecodeUTF8String(So_the) AS So_the FROM Ban_doc"
df_bandoc = pd.read_sql(query_Ban_doc, conn_libol)

df_phieumuon['ID_ban_doc'] = None
df_phieumuon['ID_ban_doc'] = df_phieumuon['So_the_ID'].apply(
                                                            lambda x: df_bandoc.loc[df_bandoc['ID'] == x, 
                                                                                    'So_the'].iloc[0] 
                                                            if not df_bandoc[df_bandoc['ID'] == x].empty else 0)
print(df_phieumuon[['So_the_ID', 'ID_ban_doc']])

C:\Users\admin\AppData\Local\Temp\ipykernel_18776\2702041677.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bandoc = pd.read_sql(query_Ban_doc, conn_libol)


  So_the_ID ID_ban_doc
0    6346.0  N97105632
1    6346.0  N97105632
2    6346.0  N97105632
3      None          0
4      None          0
5      None          0
6      None          0
7      None          0
8      None          0
9      None          0


### Kiểm tra lại so db dwh_lib

In [172]:
query_Bandoc = "SELECT ID_ban_doc FROM DIM_Ban_doc"
df_bandoc = pd.read_sql(query_Bandoc, conn_dwh_lib)
bandoc_ids = set(df_bandoc['ID_ban_doc'])
# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong ID_tai_lieu của bảng DIM_Tai_lieu hay không ?
df_phieumuon['ID_ban_doc'] = df_phieumuon['ID_ban_doc'].apply(lambda x: x if pd.notna(x) and x in bandoc_ids else 0)
print(df_phieumuon[['So_the_ID', 'ID_ban_doc']])

  So_the_ID ID_ban_doc
0    6346.0  N97105632
1    6346.0  N97105632
2    6346.0  N97105632
3      None          0
4      None          0
5      None          0
6      None          0
7      None          0
8      None          0
9      None          0


C:\Users\admin\AppData\Local\Temp\ipykernel_18776\3154858834.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bandoc = pd.read_sql(query_Bandoc, conn_dwh_lib)


# Load data

## Load data vào bảng FACT_Pieu_muon_sach

In [174]:
cursor_dwh = conn_dwh_lib.cursor()

# Lệnh INSERT cho từng hàng trong df_phieumuon
insert_query = """
INSERT INTO FACT_Phieu_muon_sach (
    ID_phieu_muon, 
    ID_tai_lieu, ID_xep_gia,
    ID_ban_doc,
    Ngay_muon, Ngay_tra,
    So_luot_gia_han, So_ngay_qua_han,
    Tien_phat, Ghi_chu
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
"""

data_to_insert = [
    (
        row['ID'], 
        row['ID_tai_lieu'], row['ID_xep_gia'], 
        row['ID_ban_doc'], 
        row['Ngay_muon'], row['Ngay_tra'],
        row['So_luot_gia_han'], row['So_ngay_qua_han'],
        row['Tien_phat'], row['Note']
    )
    for index, row in df_phieumuon.iterrows()
]
# Sử dụng executemany để chèn dữ liệu cùng lúc
cursor_dwh.executemany(insert_query, data_to_insert)
# Commit thay đổi
conn_dwh_lib.commit()
# Đóng cursor và kết nối
cursor_dwh.close()
conn_dwh_lib.close()